In [37]:
import numpy as np

In [38]:
import os

dataset_path = "/kaggle/input/datasets/adityadaveddd/nlp-dataset"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025065 - Keyur Padiya.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2024065_SDE_RESUME.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025047_GauravRajpurohit.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025059 - Kartavyakumar Patel.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025013 - Aditya Dave.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025069 (2).pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025085.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/JD.pdf


In [39]:
import os

dataset_path = "/kaggle/input/datasets/adityadaveddd/nlp-dataset"

files = sorted(os.listdir(dataset_path))

resume_count = 1

for file in files:
    old_path = os.path.join(dataset_path, file)

    # Skip directories
    if not os.path.isfile(old_path):
        continue

    # Identify JD
    if "jd" in file.lower():
        new_name = "jd.pdf"
    else:
        new_name = f"resume_{resume_count}.pdf"
        resume_count += 1

    new_path = os.path.join("/kaggle/working", new_name)

    # Copy (Kaggle input is read-only)
    with open(old_path, "rb") as f_src:
        with open(new_path, "wb") as f_dst:
            f_dst.write(f_src.read())

    print(f"{file} → {new_name}")

JD.pdf → jd.pdf
MT2024065_SDE_RESUME.pdf → resume_1.pdf
MT2025013 - Aditya Dave.pdf → resume_2.pdf
MT2025047_GauravRajpurohit.pdf → resume_3.pdf
MT2025059 - Kartavyakumar Patel.pdf → resume_4.pdf
MT2025065 - Keyur Padiya.pdf → resume_5.pdf
MT2025069 (2).pdf → resume_6.pdf
MT2025085.pdf → resume_7.pdf


In [40]:
os.listdir("/kaggle/working")

['.virtual_documents',
 'resume_5.pdf',
 'resume_2.pdf',
 'resume_7.pdf',
 'jd.pdf',
 'resume_1.pdf',
 'resume_3.pdf',
 'resume_4.pdf',
 'resume_6.pdf']

In [41]:
!pip install pdfplumber

In [42]:
import pdfplumber
import os

WORKING_DIR = "/kaggle/working"

def extract_text_from_pdf(file_path):
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                content = page.extract_text()
                if content:
                    text += content + "\n"
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return text

In [43]:
resume_texts = []
resume_files = []

jd_text = ""

for file in os.listdir(WORKING_DIR):
    path = os.path.join(WORKING_DIR, file)

    if file.startswith("resume"):
        text = extract_text_from_pdf(path)
        resume_texts.append(text)
        resume_files.append(file)

    elif file == "jd.pdf":
        jd_text = extract_text_from_pdf(path)

In [44]:
print("JD Sample:\n", jd_text[:500])
print("\n----------------------\n")
print("Resume Sample:\n", resume_texts[0][:500])

JD Sample:
 Role – MTS Intern (Member Technical Staff)
Job Summary:
We are seeking a highly motivated and talented MTS Intern to join our team. As an
Intern, you will be responsible for web development, automation, and programming
tasks. You will work on a variety of projects, collaborating with cross-functional teams
to deliver high-quality software solutions. This is an excellent opportunity to gain
practical experience and enhance your skills in a fast-paced environment.
Join us and see where you can mak

----------------------

Resume Sample:
 Keyur Sanjaykumar Padiya
International Institute of Information Technology, Bangalore
(cid:131) +91 6355100462 # keyworkmail2@gmail.com (cid:239) LinkedIn § GitHub
Education
International Institute of Information Technology, Bangalore 2025 – Present
M.Tech in Computer Science and Engineering CGPA: 3.48/4
Madhuben & Bhanubhai Patel Institute of Technology, CVM University 2020 – 2024
B.Tech in Computer Engineering CGPA: 3.3/4
Technical Skills
L

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [jd_text] + resume_texts

vectorizer = TfidfVectorizer(stop_words='english')

tfidf_matrix = vectorizer.fit_transform(documents)

In [47]:
jd_vector = tfidf_matrix[0]          # first is JD
resume_vectors = tfidf_matrix[1:]    # rest are resumes

In [48]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(jd_vector, resume_vectors)[0]

In [49]:
ranked = sorted(
    zip(resume_files, scores),
    key=lambda x: x[1],
    reverse=True
)

for i, (file, score) in enumerate(ranked, 1):
    print(f"{i}. {file} → Score: {score:.4f}")

1. resume_5.pdf → Score: 0.1187
2. resume_1.pdf → Score: 0.1051
3. resume_6.pdf → Score: 0.0972
4. resume_7.pdf → Score: 0.0859
5. resume_3.pdf → Score: 0.0726
6. resume_4.pdf → Score: 0.0685
7. resume_2.pdf → Score: 0.0600


In [50]:
def extract_name(text):
    lines = text.strip().split("\n")
    
    for line in lines[:5]:  # check first few lines
        line = line.strip()
        
        # skip empty or very short lines
        if len(line) < 3:
            continue
        
        # skip lines with numbers or keywords
        if any(char.isdigit() for char in line):
            continue
        
        if "resume" in line.lower():
            continue
        
        return line
    
    return "Unknown"

In [51]:
candidate_names = []

for text in resume_texts:
    name = extract_name(text)
    candidate_names.append(name)

In [52]:
ranked = sorted(
    zip(candidate_names, resume_files, scores),
    key=lambda x: x[2],
    reverse=True
)

for i, (name, file, score) in enumerate(ranked, 1):
    print(f"{i}. {name} ({file}) → Score: {score:.4f}")

1. Keyur Sanjaykumar Padiya (resume_5.pdf) → Score: 0.1187
2. Jainish Parmar (resume_1.pdf) → Score: 0.1051
3. Mayank Satapara (resume_6.pdf) → Score: 0.0972
4. Parva Parmar (resume_7.pdf) → Score: 0.0859
5. Gaurav Rajpurohit (resume_3.pdf) → Score: 0.0726
6. KARTAVYA PATEL (resume_4.pdf) → Score: 0.0685
7. Aditya Dave (resume_2.pdf) → Score: 0.0600
